In [1]:
import glob
import pandas as pd

In [3]:
file_paths = glob.glob("data/old/*.csv")

df_list = [
    pd.read_csv(f, usecols=["RollNo", "Name"]) for f in sorted(file_paths)
]

In [4]:
combined_df = (
    pd.concat(df_list, ignore_index=True)
    .assign(
        RollNo=lambda df: df["RollNo"].astype(str).str.strip(),
        Name=lambda df: df["Name"].astype(str).str.strip(),
    )
    .drop_duplicates(subset=["RollNo"])
    .sort_values(
        by="RollNo",
        key=lambda s: s.str[-5:].astype(int),
        ascending=True,
    )
    .reset_index(drop=True)
)

In [5]:
combined_df.to_csv("data/people.csv", index=False)

In [5]:
traffic_df = pd.read_csv("data/scrapped_data.csv")

extracted_roll = (
    "IMS" + traffic_df["Password"].astype(str).str.strip().str[:-2].str[-5:]
)

roll_to_name = dict(zip(combined_df["RollNo"], combined_df["Name"].str.title()))

traffic_df["Name"] = (
    extracted_roll.map(roll_to_name).fillna(traffic_df["Name"].str.title())
)

traffic_df.to_csv("data/scrapped_updated.csv", index=False)

In [6]:
updated = pd.read_csv("data/scrapped_updated.csv")

updated["RollNo"] = (
    "IMS"
    + updated["Password"].dropna().astype(str).str.strip().str[:-2].str[-5:]
)

updated["Name"] = updated["Name"].astype(str).str.strip().str.title()

missing_df = (
    combined_df[~combined_df["RollNo"].isin(updated["RollNo"])]
    .copy()
    .assign(Name=lambda df: df["Name"].astype(str).str.strip().str.title())
    .sort_values(by="RollNo", key=lambda s: s.str[-5:].astype(int))
    .reset_index(drop=True)
)

new_rows = pd.DataFrame(
    {
        "RollNo": missing_df["RollNo"],
        "Name": missing_df["Name"],
        "Username": missing_df["Name"].str.split().str[0].str.lower() + "23",
        "Password": pd.NA,
        "Upload": 0.0,
        "Download": 0.0,
        "Total traffic": 0.0,
    }
)

updated_complete = (
    pd.concat([updated, new_rows], ignore_index=True)
    .sort_values(by="RollNo", key=lambda s: s.str[-5:].astype(int))
    .drop(columns=["RollNo"])
    .reset_index(drop=True)
)

updated_complete.to_csv("data/scrapped_updated_complete.csv", index=False)
missing_df.to_csv("data/missing_students.csv", index=False)

In [4]:
people = pd.read_csv("data/people.csv")
scrapped = pd.read_csv("data/scrapped_updated_complete.csv")

people["clean_name"] = people["Name"].astype(str).str.strip().str.lower()
scrapped["clean_name"] = scrapped["Name"].astype(str).str.strip().str.lower()

name_to_roll = dict(zip(people["clean_name"], people["RollNo"]))
roll_from_name = scrapped["clean_name"].map(name_to_roll)

roll_from_pwd = (
    "IMS"
    + scrapped["Password"].dropna().astype(str).str.strip().str[:-2].str[-5:]
)

scrapped["Roll"] = roll_from_name.fillna(roll_from_pwd)
scrapped["Name"] = scrapped["Name"].astype(str).str.strip().str.title()

result = (
    scrapped.sort_values(by="Roll", key=lambda s: s.str[-5:].astype(int))
    .reindex(columns=["Roll", "Name", "Username", "Password"])
    .rename(columns={"Username": "username", "Password": "password"})
    .reset_index(drop=True)
)

result.to_csv("data/students_credentials.csv", index=False)